# Conservative Hard-Sparsity Stable Drift Prior on GOLDEN

This notebook prototypes the sparsity-preserving construction for CT drift priors on the full `GOLDEN` accepted Stage 4 state. It benchmarks the dynamic endogenous drift block only; time-invariant/exogenous retained states stay outside the joint CT dynamic block. This is a prior-level benchmark only: no pipeline evals or inference runs are triggered.

In [1]:

from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
from scipy.linalg import expm


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "apps" / "data-pipeline").exists() and (current / "data").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate repository root from notebook working directory")


REPO_ROOT = find_repo_root(Path.cwd())
GOLDEN_RUN = REPO_ROOT / "data" / ".private" / "GOLDEN" / "run"
STAGE1B_PATH = GOLDEN_RUN / "stage-1b.json"
MEGAPROMPT_PATH = GOLDEN_RUN / "stage-4-megaprompt.json"

stage1b = json.loads(STAGE1B_PATH.read_text())
megaprompt = json.loads(MEGAPROMPT_PATH.read_text())
accepted = megaprompt["accepted"]
causal_spec = stage1b["causal_spec"]
resolved_priors = {
    prior["parameter"]: prior
    for prior in accepted["resolved_priors"]
    if prior is not None
}

all_state_names = list(causal_spec["estimation"]["state_order"])
constructs = {construct["name"]: construct for construct in causal_spec["latent"]["constructs"]}
latent_names = [
    name
    for name in all_state_names
    if constructs[name].get("role") == "endogenous"
    and constructs[name].get("temporal_status") == "time_varying"
]
n_latent = len(latent_names)
latent_index = {name: idx for idx, name in enumerate(latent_names)}

drift_mask = np.eye(n_latent, dtype=bool)
for edge in causal_spec["latent"]["edges"]:
    cause = edge["cause"]
    effect = edge["effect"]
    parameter = f"beta_{cause}_{effect}"
    if cause in latent_index and effect in latent_index and parameter in resolved_priors:
        drift_mask[latent_index[effect], latent_index[cause]] = True

diag_mask = np.eye(n_latent, dtype=bool)
allowed_offdiag_mask = drift_mask & ~diag_mask
structural_zero_mask = (~drift_mask) & ~diag_mask
allowed_positions = [
    (row, col)
    for row in range(n_latent)
    for col in range(n_latent)
    if row != col and drift_mask[row, col]
]


def duration_to_days(value: str) -> float:
    if value.endswith("d"):
        return float(value[:-1])
    if value.endswith("h"):
        return float(value[:-1]) / 24.0
    raise ValueError(f"Unsupported model clock {value!r}")


model_dt_days = duration_to_days(causal_spec["measurement"]["model_clock"])

print(f"Golden run: {GOLDEN_RUN.relative_to(REPO_ROOT)}")
print(f"Dynamic drift block ({n_latent} states): {', '.join(latent_names)}")
print("Excluded retained states:")
for name in all_state_names:
    if name not in latent_index:
        construct = constructs[name]
        print(f"  {name}: {construct.get('role')} / {construct.get('temporal_status')}")
print(f"Model interval: {model_dt_days:g} day")
print(f"Allowed off-diagonal dynamic drift entries: {int(allowed_offdiag_mask.sum())}")
print(f"Structural-zero off-diagonal dynamic entries: {int(structural_zero_mask.sum())}")
print("Allowed dynamic topology edges:")
for row, col in allowed_positions:
    print(f"  {latent_names[col]} -> {latent_names[row]}")


Golden run: data/.private/GOLDEN/run
Dynamic drift block (9 states): sleep_quality, sleep_duration, screen_time, evening_screen_use, screen_content_type, social_media_use, stress, mental_health, bedtime_delay
Excluded retained states:
  chronotype: exogenous / time_invariant
Model interval: 1 day
Allowed off-diagonal dynamic drift entries: 11
Structural-zero off-diagonal dynamic entries: 61
Allowed dynamic topology edges:
  sleep_duration -> sleep_quality
  stress -> sleep_quality
  mental_health -> sleep_quality
  bedtime_delay -> sleep_duration
  stress -> screen_time
  mental_health -> screen_time
  screen_time -> evening_screen_use
  screen_content_type -> social_media_use
  mental_health -> stress
  stress -> mental_health
  evening_screen_use -> bedtime_delay


## Construction

For each draw, allowed off-diagonal entries are sampled from the accepted GOLDEN scalar effect priors. Forbidden entries remain exactly zero. Each diagonal is then set to the negative row-wise absolute off-diagonal mass plus an independent base decay and a fixed margin. This is a Gershgorin-style strict diagonal-dominance guarantee: every sampled dynamic drift matrix is stable without a rejection step or runtime stability penalty.

In [2]:

def sample_prior_1d(prior: dict, rng: np.random.Generator, n_draws: int) -> np.ndarray:
    family = prior["distribution"]
    params = prior["params"]
    if family == "Beta":
        return rng.beta(params["alpha"], params["beta"], size=n_draws)
    if family == "Uniform":
        return rng.uniform(params["lower"], params["upper"], size=n_draws)
    if family == "Normal":
        return rng.normal(params["mu"], params["sigma"], size=n_draws)
    raise ValueError(f"Unsupported drift prior family {family!r}")


def sample_current_scalar_prior(rng: np.random.Generator, n_draws: int) -> np.ndarray:
    draws = np.zeros((n_draws, n_latent, n_latent), dtype=float)
    for idx, name in enumerate(latent_names):
        rho = sample_prior_1d(resolved_priors[f"rho_{name}"], rng, n_draws)
        rho = np.clip(rho, 1e-8, 1.0 - 1e-8)
        draws[:, idx, idx] = np.log(rho) / model_dt_days
    for row, col in allowed_positions:
        beta = sample_prior_1d(resolved_priors[f"beta_{latent_names[col]}_{latent_names[row]}"], rng, n_draws)
        draws[:, row, col] = beta / model_dt_days
    return draws


def interval_response_samples(draws: np.ndarray, dt_days: float, max_draws: int = 1000) -> np.ndarray:
    n = min(max_draws, draws.shape[0])
    return np.stack([expm(draws[i] * dt_days) for i in range(n)], axis=0)


def summarize_drift_samples(name: str, draws: np.ndarray, elapsed_seconds: float) -> dict[str, float | str]:
    eigvals = np.linalg.eigvals(draws)
    max_real = eigvals.real.max(axis=1)
    zero_abs = np.abs(draws[:, structural_zero_mask])
    allowed_abs = np.abs(draws[:, allowed_offdiag_mask])
    response = interval_response_samples(draws, model_dt_days)
    absent_response_abs = np.abs(response[:, structural_zero_mask])
    allowed_response_abs = np.abs(response[:, allowed_offdiag_mask])
    return {
        "name": name,
        "draws": draws.shape[0],
        "stable_rate": float(np.mean(max_real < 0.0)),
        "margin_q05": float(np.quantile(-max_real, 0.05)),
        "diag_mean": float(np.mean(np.diagonal(draws, axis1=1, axis2=2))),
        "absent_a_q90": float(np.quantile(zero_abs, 0.90)),
        "absent_a_max": float(np.max(zero_abs)),
        "allowed_a_q90": float(np.quantile(allowed_abs, 0.90)),
        "absent_response_q90": float(np.quantile(absent_response_abs, 0.90)),
        "allowed_response_q90": float(np.quantile(allowed_response_abs, 0.90)),
        "seconds": float(elapsed_seconds),
    }


def print_summary_table(rows: list[dict[str, float | str]]) -> None:
    columns = [
        ("name", "prior"),
        ("draws", "draws"),
        ("stable_rate", "stable"),
        ("margin_q05", "margin q05"),
        ("diag_mean", "diag mean"),
        ("absent_a_q90", "absent |A| q90"),
        ("absent_a_max", "absent |A| max"),
        ("allowed_a_q90", "allowed |A| q90"),
        ("absent_response_q90", "absent |exp(AΔ)| q90"),
        ("allowed_response_q90", "allowed |exp(AΔ)| q90"),
        ("seconds", "seconds"),
    ]
    widths = []
    for key, label in columns:
        values = [label]
        for row in rows:
            value = row[key]
            values.append(str(value) if isinstance(value, str) else f"{value:.4g}")
        widths.append(max(len(v) for v in values))
    header = "  ".join(label.ljust(width) for (_, label), width in zip(columns, widths))
    print(header)
    print("  ".join("-" * width for width in widths))
    for row in rows:
        parts = []
        for (key, _label), width in zip(columns, widths):
            value = row[key]
            text = str(value) if isinstance(value, str) else f"{value:.4g}"
            parts.append(text.ljust(width))
        print("  ".join(parts))


In [3]:

def sample_hard_sparse_stable_prior(
    rng: np.random.Generator,
    n_draws: int,
    *,
    stability_margin: float = 0.05,
    base_decay_shape: float = 6.0,
) -> np.ndarray:
    draws = np.zeros((n_draws, n_latent, n_latent), dtype=float)
    for row, col in allowed_positions:
        beta = sample_prior_1d(
            resolved_priors[f"beta_{latent_names[col]}_{latent_names[row]}"],
            rng,
            n_draws,
        )
        draws[:, row, col] = beta / model_dt_days

    scalar_reference = sample_current_scalar_prior(rng, min(n_draws, 2000))
    base_decay_mean = float(
        np.median(-np.diagonal(scalar_reference, axis1=1, axis2=2))
    )
    base_decay = rng.gamma(
        shape=base_decay_shape,
        scale=base_decay_mean / base_decay_shape,
        size=(n_draws, n_latent),
    )
    row_abs_mass = np.sum(np.abs(draws), axis=2)
    draws[:, np.arange(n_latent), np.arange(n_latent)] = -(
        base_decay + row_abs_mass + stability_margin
    )
    return draws


## Benchmark

In [4]:

N_DRAWS = 5000
rng = np.random.default_rng(20260428)

start = time.perf_counter()
scalar_draws = sample_current_scalar_prior(rng, N_DRAWS)
scalar_elapsed = time.perf_counter() - start

start = time.perf_counter()
hard_draws = sample_hard_sparse_stable_prior(rng, N_DRAWS)
hard_elapsed = time.perf_counter() - start

rows = [
    summarize_drift_samples("accepted scalar prior", scalar_draws, scalar_elapsed),
    summarize_drift_samples("hard sparse stable", hard_draws, hard_elapsed),
]
print_summary_table(rows)


prior                  draws  stable  margin q05  diag mean  absent |A| q90  absent |A| max  allowed |A| q90  absent |exp(AΔ)| q90  allowed |exp(AΔ)| q90  seconds 
---------------------  -----  ------  ----------  ---------  --------------  --------------  ---------------  --------------------  ---------------------  --------
accepted scalar prior  5000   1       3.569       -4.741     0               0               1.5              0.003515              0.01414                0.001459
hard sparse stable     5000   1       1.859       -5.916     0               0               1.5              0.001073              0.01975                0.003026


## SCC View

In [5]:

import networkx as nx

G = nx.DiGraph()
G.add_nodes_from(latent_names)
for row, col in allowed_positions:
    G.add_edge(latent_names[col], latent_names[row])

sccs = [sorted(component) for component in nx.strongly_connected_components(G)]
feedback_sccs = [component for component in sccs if len(component) > 1]
print("Strongly connected components:")
for component in sccs:
    print(f"  {component}")
print("Feedback SCCs needing a joint drift prior:")
for component in feedback_sccs:
    print(f"  {component}")


Strongly connected components:
  ['sleep_quality']
  ['sleep_duration']
  ['bedtime_delay']
  ['evening_screen_use']
  ['screen_time']
  ['social_media_use']
  ['screen_content_type']
  ['mental_health', 'stress']
Feedback SCCs needing a joint drift prior:
  ['mental_health', 'stress']


## Reading

The hard construction preserves causal topology exactly and moves instability prevention into the prior geometry. The cost is visible in the diagonal: strong accepted GOLDEN effects force additional decay, so decay speed is no longer independent of coupling strength.